<a href="https://colab.research.google.com/github/Normg530/cybersecurity-login-anomaly/blob/main/CapstoneProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import hashlib

In [2]:
from pathlib import Path

In [4]:
df = pd.read_csv("cybersecurity_login_dataset.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   login_attempts      600 non-null    int64
 1   failed_logins       600 non-null    int64
 2   unusual_login_hour  600 non-null    int64
 3   new_ip_address      600 non-null    int64
 4   suspicious          600 non-null    int64
dtypes: int64(5)
memory usage: 23.6 KB


In [6]:
df.columns

Index(['login_attempts', 'failed_logins', 'unusual_login_hour',
       'new_ip_address', 'suspicious'],
      dtype='object')

In [7]:
df.isnull().sum()

,0
login_attempts,0
failed_logins,0
unusual_login_hour,0
new_ip_address,0
suspicious,0


In [8]:
df.head()

,login_attempts,failed_logins,unusual_login_hour,new_ip_address,suspicious
0,4,0,1,0,0
1,5,5,0,0,0
2,14,0,0,0,0
3,8,8,0,0,0
4,21,17,1,0,1


In [9]:
df.shape

(600, 5)

In [10]:
# remove spaces
df.columns = df.columns.str.strip()

In [11]:
# calculate the total falied logins
total_failed_logins = df['failed_logins'].sum()

In [12]:
print(total_failed_logins)

3845


In [13]:
# failed logins by ip address
failed_logins_by_ip = df.groupby('new_ip_address')['failed_logins'].sum()

In [14]:
print(failed_logins_by_ip)

new_ip_address
0    1724
1    2121
Name: failed_logins, dtype: int64


In [54]:
df['unusual_login_hour'].value_counts()

,count
unusual_login_hour,
1,307
0,293


In [55]:
df.groupby('unusual_login_hour')['failed_logins'].sum()

,failed_logins
unusual_login_hour,
0,1785
1,2060


In [56]:
df['new_ip_address'].value_counts()

,count
new_ip_address,
1,327
0,273


In [15]:
# most suspicious ip address
suspicious_logins = df[df["suspicious"] == 1]

print(suspicious_logins)

     login_attempts  failed_logins  unusual_login_hour  new_ip_address  \
4                21             17                   1               0   
5                19              8                   0               0   
7                25             10                   0               0   
8                 4              2                   1               1   
9                24             14                   0               1   
..              ...            ...                 ...             ...   
592               5              1                   1               1   
593              22             15                   1               1   
596              25             14                   1               0   
598              24             14                   0               1   
599              24             20                   0               1   

     suspicious  
4             1  
5             1  
7             1  
8             1  
9             1  
.. 

In [16]:
most_suspicious = (
    df[df["suspicious"] == 1]
    .sort_values("failed_logins", ascending=False)
)

print(most_suspicious.head(10))

     login_attempts  failed_logins  unusual_login_hour  new_ip_address  \
434              25             25                   0               1   
429              25             23                   1               1   
117              24             23                   0               1   
28               24             23                   1               0   
483              22             22                   0               0   
377              23             22                   0               1   
59               22             22                   1               0   
519              23             21                   1               0   
208              21             21                   1               0   
516              21             21                   1               1   

     suspicious  
434           1  
429           1  
117           1  
28            1  
483           1  
377           1  
59            1  
519           1  
208           1  
516  

In [17]:
# repeated failed logins
repeated_failed = df[df["failed_logins"] >= 5]

print(repeated_failed)

     login_attempts  failed_logins  unusual_login_hour  new_ip_address  \
1                 5              5                   0               0   
3                 8              8                   0               0   
4                21             17                   1               0   
5                19              8                   0               0   
6                14              5                   1               0   
..              ...            ...                 ...             ...   
588              13              6                   0               1   
593              22             15                   1               1   
596              25             14                   1               0   
598              24             14                   0               1   
599              24             20                   0               1   

     suspicious  
1             0  
3             0  
4             1  
5             1  
6             0  
.. 

In [18]:
df.sort_values("failed_logins", ascending=False).head(10)

,login_attempts,failed_logins,unusual_login_hour,new_ip_address,suspicious
434,25,25,0,1,1
28,24,23,1,0,1
429,25,23,1,1,1
117,24,23,0,1,1
377,23,22,0,1,1
59,22,22,1,0,1
483,22,22,0,0,1
566,21,21,1,1,1
315,23,21,1,0,1
313,24,21,0,1,1


In [19]:
# unusual login hour
unusual_activity = df[df["unusual_login_hour"] == 1]

print(unusual_activity)

     login_attempts  failed_logins  unusual_login_hour  new_ip_address  \
0                 4              0                   1               0   
4                21             17                   1               0   
6                14              5                   1               0   
8                 4              2                   1               1   
10               18              9                   1               0   
..              ...            ...                 ...             ...   
591              20              1                   1               0   
592               5              1                   1               1   
593              22             15                   1               1   
596              25             14                   1               0   
597               4              2                   1               0   

     suspicious  
0             0  
4             1  
6             0  
8             1  
10            1  
.. 

In [20]:
high_risk = df[
    (df["failed_logins"] >= 5) &
    (df["unusual_login_hour"] == 1) &
    (df["new_ip_address"] == 1)
]

print(high_risk)

     login_attempts  failed_logins  unusual_login_hour  new_ip_address  \
21                8              8                   1               1   
25               13              9                   1               1   
34               16              5                   1               1   
35               18              6                   1               1   
36               21             11                   1               1   
..              ...            ...                 ...             ...   
563              15             15                   1               1   
566              21             21                   1               1   
572              15              6                   1               1   
583              22             15                   1               1   
593              22             15                   1               1   

     suspicious  
21            1  
25            1  
34            1  
35            1  
36            1  
.. 

In [21]:
# need to add at least 3 toolkit components
# first one is password checker
password = "password123"
hash_password = hashlib.sha256(password.encode()).hexdigest()

print(hash_password)

ef92b778bafe771e89245b89ecbc08a44a4e166c06659911881f383d4473e94f


In [22]:
print('Total log records:', len(df))

Total log records: 600


In [23]:
df['login_attempts'].sum()

np.int64(7914)

In [24]:
df["failed_logins"].sum()

np.int64(3845)

In [25]:
# i also used a file intergrity checker
import hashlib

def calculate_hash(filename):
    sha256 = hashlib.sha256()

    with open(filename, "rb") as file:
        while chunk := file.read(4096):
            sha256.update(chunk)

    return sha256.hexdigest()

In [27]:
# Choose the file to keep track of
filename = "example.txt"

# Create a dummy file for demonstration if it doesn't exist
if not Path(filename).exists():
    with open(filename, "w") as f:
        f.write("This is a test file for integrity checking.")

# Create the original hash
original_hash = calculate_hash(filename)

print("Original SHA-256 hash:")
print(original_hash)

# Check the file again
current_hash = calculate_hash(filename)

if current_hash == original_hash:
    print("✅ File integrity verified - file has not changed.")
else:
    print("⚠️ WARNING - file has been changed!")

Original SHA-256 hash:
9fdf0314dfd4cf9ba59f707f9eb3a9a44907e5fd394382536b3c081d0308064a
✅ File integrity verified - file has not changed.


In [ ]:
# example
with open("example.txt", "w") as file:
    file.write("This is my original file."

In [28]:

def calculate_hash(filename):
    sha256 = hashlib.sha256()

    with open(filename, "rb") as file:
        while chunk := file.read(4096):
            sha256.update(chunk)

    return sha256.hexdigest()

original_hash = calculate_hash("example.txt")

print("Original hash:")
print(original_hash)

Original hash:
9fdf0314dfd4cf9ba59f707f9eb3a9a44907e5fd394382536b3c081d0308064a


In [29]:
with open("example.txt", "w") as file:
    file.write("This file has been changed!")

In [30]:
new_hash = calculate_hash("example.txt")

print("New hash:")
print(new_hash)

if original_hash == new_hash:
    print("File is unchanged.")
else:
    print("⚠️ File has been modified!")

New hash:
1b0c6bb8117d5c580bc6c96edffc77f81d27d3cc8dea131762d3881cac6ce796
⚠️ File has been modified!


In [ ]:
# will try to use a basic phising detector, but there is no email column
# then must use the suspicious column ...

In [32]:
df['suspicious'].value_counts()

,count
suspicious,
1,367
0,233


In [34]:
print(df['suspicious'].value_counts())

suspicious
1    367
0    233
Name: count, dtype: int64


In [41]:
features = [
    "login_attempts",
    "failed_logins",
    "unusual_login_hour",
    "new_ip_address"
]

X = df[features]
y = df["suspicious"]

In [42]:
# need to split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [43]:
# imort standard scaler
from sklearn.preprocessing import StandardScaler

In [44]:
# scale the fetures
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [45]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [47]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8833333333333333

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86        47
           1       0.94      0.86      0.90        73

    accuracy                           0.88       120
   macro avg       0.88      0.89      0.88       120
weighted avg       0.89      0.88      0.88       120



In [48]:
# we can give the model numbers to see if they are suspicious
new_login = [[20, 15, 1, 1]]

prediction = model.predict(new_login)

if prediction[0] == 1:
    print("⚠️ Suspicious login detected!")
else:
    print("✅ Login appears normal.")

⚠️ Suspicious login detected!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [50]:
new_login = [[10, 10,1,0]]

prediction = model.predict(new_login)

if prediction[0] == 1:
    print("⚠️ Suspicious login detected!")
else:
    print("✅ Login appears normal.")

⚠️ Suspicious login detected!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [51]:
new_login = [[5, 5,1,0]]

prediction = model.predict(new_login)

if prediction[0] == 1:
    print("⚠️ Suspicious login detected!")
else:
    print("✅ Login appears normal.")

✅ Login appears normal.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [52]:
# caesar cipher
def caesar_cipher(text, shift):
    result = ""

    for char in text:
        if char.isalpha():
            start = ord('A') if char.isupper() else ord('a')

            new_char = chr(
                (ord(char) - start + shift) % 26 + start
            )

            result += new_char
        else:
            result += char

    return result

In [53]:
message = "Hello World"
shift = 3

encrypted = caesar_cipher(message, shift)

print("Original:", message)
print("Encrypted:", encrypted)

Original: Hello World
Encrypted: Khoor Zruog
